# COROSred: Confidence-Routed Selective Re-Diffusion
This notebook executes the training pipeline for **COROSred**.


In [1]:
import os
import sys
import time
import gc
import yaml
from pathlib import Path

# Ensure working directory is project root
project_root = Path.cwd()
while not (project_root / "COROSred").exists() and project_root.parent != project_root:
    project_root = project_root.parent
os.chdir(project_root)
sys.path.insert(0, str(project_root))

import mlx.core as mx
from COROSred.model import COROSredTransformer
from COROSred.training import TelosMLXCOROSredTrainer

def run_corosred_training(config_path, base_ar_weights=None, resume_from=None, resume_step=0):
    print("=" * 85)
    print("STARTING COROSred RUN: " + str(config_path))
    print("=" * 85)
    with open(config_path, "r") as f:
        cfg = yaml.safe_load(f)
    
    model = COROSredTransformer(**cfg["model"])
    model.set_dtype(mx.bfloat16)
    
    if base_ar_weights:
        print(f"  [Init] Bootstrapping AR backbone from {base_ar_weights}")
        model.load_weights(base_ar_weights, strict=False)

    if resume_from:
        print(f"  [Resume] Loading exact head/model weights from {resume_from}")
        model.load_weights(resume_from, strict=False)
    
    trainer = TelosMLXCOROSredTrainer(model, cfg)
    trainer.train(resume_step=resume_step)
    
    del model, trainer
    gc.collect()
    mx.clear_cache()
    print("FINISHED RUN: " + str(config_path) + "\n")

In [ ]:
# Phase A execution: Train Reliability head on 12.5M AR Backbone
start_time = time.time()

# Note: Replace `base_ar_weights` with the actual path to your completed AR 12.5M .safetensors artifact
ar_baseline_weights_path = "checkpoints/ar/12_5m/model.safetensors" 

run_corosred_training(
    config_path="configs/corosred/phase_a.yaml",
    base_ar_weights=ar_baseline_weights_path
)

total_elapsed = (time.time() - start_time) / 3600.0
print("=" * 85)
print(f"COROSred Phase A completed in {total_elapsed:.2f} hours!")
print("=" * 85)